### Import Dependencies

In [1]:
import openai
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from dotenv import find_dotenv, load_dotenv
from langsmith import Client

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path)

ls_client = Client()

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
all_points = qdrant_client.scroll(
    collection_name="toys_and_games_small_sample",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [6]:
all_points

([Record(id=0, payload={'images': 'https://m.media-amazon.com/images/I/51TgjhgWjCS._AC_.jpg', 'parent_asin': 'B09X9XW42H', 'all_description': 'Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is 

In [7]:
all_points[0][0].payload

{'images': 'https://m.media-amazon.com/images/I/51TgjhgWjCS._AC_.jpg',
 'parent_asin': 'B09X9XW42H',
 'all_description': 'Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you o

In [9]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["all_description"]} for data in all_points[0]]

In [10]:
all_context

[{'id': 'B09X9XW42H',
  'text': 'Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. This Mothman plushie is looking for a loving and magical home rig

In [19]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 100 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 40 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [12]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 100 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions,

In [20]:
response = openai.chat.completions.create(
    model="gpt-6-luna",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

In [14]:
print(response.choices[0].message.content)

[
  {
    "reasoning": "Both products are plush toys, and their descriptions specify their respective sizes, so they can be compared directly.",
    "question": "Could you compare the size of the Mothman plush and the Sailor Moon plush?",
    "chunk_ids": ["B09X9XW42H", "B00O7A1I64"],
    "answer_example": "The Mothman plush is approximately 8 inches tall and 9 inches wide. The Sailor Moon plush is 7 inches tall.",
    "reasoning": "Both products are plush toys, and their descriptions specify their respective sizes, so they can be compared directly."
  },
  {
    "reasoning": "These are two distinct RC vehicles, and their product descriptions specify different play features and battery arrangements.",
    "question": "Which RC vehicle comes with rechargeable batteries, and which one has a turbo drifting feature?",
    "chunk_ids": ["B0BCPLBL8Z", "B08V616T5G"],
    "answer_example": "The Kidstinct RC tank comes with two rechargeable batteries. The Jada Toys Ford Mustang has a turbo butt

In [15]:
response.usage

CompletionUsage(completion_tokens=3703, prompt_tokens=29570, total_tokens=33273, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=29567, cached_tokens=0, image_tokens=None, text_tokens=None))

In [21]:
json_output = json.loads(response.choices[0].message.content)
json_output

[{'reasoning': "The product description specifies the plush's approximate height and width.",
  'question': 'How big is the Gothic Mothman plush?',
  'chunk_ids': ['B09X9XW42H'],
  'answer_example': 'It is approximately 8 inches tall and 9 inches wide.'},
 {'reasoning': "The listing gives the monster truck's scale and recommended minimum age.",
  'question': 'What scale is the Bone Shaker monster truck, and what age is it for?',
  'chunk_ids': ['B09G7K3JWQ'],
  'answer_example': 'It is a 1:24-scale vehicle and is recommended for children ages 3 and up.'},
 {'reasoning': 'The kit description lists both the number of sticker templates and the included tools and materials.',
  'question': 'What comes with the transportation diamond painting kit?',
  'chunk_ids': ['B0BQJGV4ZM'],
  'answer_example': 'It includes 12 numbered sticker templates, colored diamonds, two pens, clay, a plastic tray, 17 PVC bags, and instructions.'},
 {'reasoning': 'The pinata listing states its filling capacity and

In [22]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [24]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 20
Multiple chunk questions: 11
Impossible questions: 0


In [29]:
points = qdrant_client.scroll(
    collection_name="toys_and_games_small_sample",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B09G7K3JWQ")
            )
        ]
    )
)[0]

In [26]:
points

[Record(id=0, payload={'images': 'https://m.media-amazon.com/images/I/51TgjhgWjCS._AC_.jpg', 'parent_asin': 'B09X9XW42H', 'all_description': 'Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is p

In [31]:
def get_description(parent_asin: str) -> str:

    points = qdrant_client.scroll(
        collection_name="toys_and_games_small_sample",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["all_description"]

In [32]:
get_description("B09G7K3JWQ")

'Hot Wheels Monster Truck 1:24 Scale 2022 Bone Shaker It All Vehicle with Giant Wheels for Kids Age 3 to 8 Years Old Great Gift Toy Trucks Large Scale Designed in 1:24 scale with durable die-cast metal bodies, Hot Wheels Monster Trucks are ready to for outrageous action!\u200b\u200b Giant wheels and rad details amp up the collecting and crashing, smashing and stunt play! The vehicle package features cool stats on back giving details about strength, crash attack, motor-vation and more! Rev up imagination and storytelling with unique toy Monster Trucks that deliver mind-blowing features -- fans will want them all! Makes a great gift for kids and Monster Trucks fans ages 3 years and older! The Hot Wheels Monster Trucks 1:24 scale die-cast trucks are THE baddest trucks ever built for competition and ultimate dominance! The outrageous vehicles are ready to square off in head-to-head battles with an oversized body and GIANT wheels! Every toy truck package includes fun stats for the monster t

In [38]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [39]:
ls_client = Client()

In [40]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [41]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"],
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )